# Étude de cas : ventes et livraisons

Une entreprise vend des produits en ligne (épicerie, frais, boissons, hygiène, maison, électronique, livres, bricolage, collection) et les livre elle-même, depuis ses entrepôts de la région parisienne, avec sa propre flotte de véhicules.

### Les fichiers (`data/` dans le projet)

| Fichier | Contenu |
|---|---|
| `produits.csv` | le catalogue : `produit_id, nom, categorie, sous_categorie, prix, poids_kg` |
| `commandes.csv` | le journal des commandes, une ligne par événement : `commande_id, client_id, horodatage, evenement, livreur_id, detail` |
| `vehicules/<vehicule>/<AAAA-MM>.csv` | la trace GPS de chaque véhicule, **un point par minute** : `timestamp, livreur_id, lon, lat, vitesse_kmh, dans_vehicule, charge_kg` |
| `communes.csv` | les communes de Paris et de la petite couronne et les 20 arrondissements de Paris, avec leur centre : `commune, code_postal, lon, lat` |
| `communes.geojson` | les contours des mêmes communes (propriétés `commune`, `code_postal`) |

### Le journal des commandes

Une commande apparaît plusieurs fois : une ligne par étape.

| `evenement` | Sens | `livreur_id` | `detail` |
|---|---|---|---|
| `commande` | le client passe commande | — | les produits : `P0201x1\|P0033x4` (produit `x` quantité) |
| `chargee` | la commande est chargée dans un véhicule | le livreur | — |
| `livree` | le client signe ; horodatage à la seconde | le livreur | — |
| `echec` | la livraison n'a pas pu se faire | le livreur | le motif |
| `reportee` | chargée mais pas tentée : rapportée à l'entrepôt en fin de service | le livreur | — |
| `retour` | la commande est abandonnée et retournée à l'entrepôt | le livreur | — |

### Les traces GPS

Un livreur conduit un seul véhicule par jour. `dans_vehicule` vaut `false` quand le livreur est descendu du véhicule ; `charge_kg` est le poids transporté.

### Exécution locale

Sélectionnez l'environnement `.venv/bin/python` (Python 3.11) comme noyau dans votre éditeur, puis exécutez la cellule 1.1.
Spark tourne sur le Mac en mode `local[*]` ; les conteneurs Spark et Jupyter ne sont pas nécessaires.
Les instructions d'installation sont dans `README.md` à la racine du projet.

| Adresse | Service |
|---|---|
| http://localhost:4040 | Spark UI de la session locale (ou un port suivant si occupé) |
| `mongodb://localhost:27017` | MongoDB, si démarré avec `docker compose up -d mongo` |

## 1) Charger la donnée

1.1) La session Spark et les fonctions fournies.

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("livraisons")
         .master("spark://spark-master:7077")
         .getOrCreate())
DATA = "/work/data"
RESULTATS = "/work/resultats"


def haversine(lon1, lat1, lon2, lat2):
    """Distance en km entre deux points : noms de colonnes ou colonnes, en degrés."""
    lon1, lat1, lon2, lat2 = [F.col(c) if isinstance(c, str) else c for c in (lon1, lat1, lon2, lat2)]
    dlon = F.radians(lon2 - lon1)
    dlat = F.radians(lat2 - lat1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(a))

1.2) Définissez un schéma explicite (`StructType`) pour chacun des 4 fichiers CSV, puis chargez `produits.csv`, `commandes.csv`, `communes.csv` et **tous** les fichiers GPS en une seule lecture (`vehicules/*/*.csv`). Les horodatages doivent être de type `timestamp`, `dans_vehicule` de type `boolean`, les nombres de type `double`.

Affichez le nombre de lignes de chaque DataFrame.



In [14]:
from pyspark.sql import types as T

schema_produits = T.StructType([
    T.StructField("produit_id", T.StringType(), False),
    T.StructField("nom", T.StringType(), False),
    T.StructField("categorie", T.StringType(), False),
    T.StructField("sous_categorie", T.StringType(), False),
    T.StructField("prix", T.DoubleType(), False),
    T.StructField("poids_kg", T.DoubleType(), False)
])

print("schema_produits = ", schema_produits)

schema_commandes = T.StructType([
    T.StructField("commande_id", T.StringType(), False),
    T.StructField("client_id", T.StringType(), False),
    T.StructField("horodatage", T.TimestampType(), False),
    T.StructField("evenement", T.StringType(), False),
    T.StructField("livreur_id", T.StringType(), False),
    T.StructField("detail", T.StringType(), False)
])

print("schema_commandes = ", schema_commandes)

schema_communes = T.StructType([
    T.StructField("commune", T.StringType(), False),
    T.StructField("code_postal", T.IntegerType(), False),
    T.StructField("lon", T.DoubleType(), False),
    T.StructField("lat", T.DoubleType(), False)
])

print("schema_communes = ", schema_communes)

schema_gps = T.StructType([
    T.StructField("timestamp", T.TimestampType(), False),
    T.StructField("livreur_id", T.StringType(), False),
    T.StructField("lon", T.DoubleType(), False),
    T.StructField("lat", T.DoubleType(), False),
    T.StructField("vitesse_kmh", T.DoubleType(), False),
    T.StructField("dans_vehicule", T.BooleanType(), False),
    T.StructField("charge_kg", T.DoubleType(), False)
])
print("schema_gps = ", schema_gps)
    

schema_produits =  StructType([StructField('produit_id', StringType(), False), StructField('nom', StringType(), False), StructField('categorie', StringType(), False), StructField('sous_categorie', StringType(), False), StructField('prix', DoubleType(), False), StructField('poids_kg', DoubleType(), False)])
schema_commandes =  StructType([StructField('commande_id', StringType(), False), StructField('client_id', StringType(), False), StructField('horodatage', TimestampType(), False), StructField('evenement', StringType(), False), StructField('livreur_id', StringType(), False), StructField('detail', StringType(), False)])
schema_communes =  StructType([StructField('commune', StringType(), False), StructField('code_postal', IntegerType(), False), StructField('lon', DoubleType(), False), StructField('lat', DoubleType(), False)])
schema_gps =  StructType([StructField('timestamp', TimestampType(), False), StructField('livreur_id', StringType(), False), StructField('lon', DoubleType(), False),

In [15]:
schemas = {
    "produits": {
        "schema": schema_produits,
        "path": f"{DATA}/produits.csv"
        },
    "commandes": {
        "schema": schema_commandes,
        "path": f"{DATA}/commandes.csv"
        },
    "communes": {
        "schema": schema_communes,
        "path": f"{DATA}/communes.csv"
        },
    "gps": {
        "schema": schema_gps,
        "path": f"{DATA}/vehicules/*/*.csv"
    }
    
}

dataframes = {}

for nom, config in schemas.items():
    path = config["path"]
    dataframes[nom] = spark.read.csv(path, header=True, schema=config["schema"])
    print(f"Le fichier {nom} contient {dataframes[nom].count()} lignes.")

Le fichier produits contient 233 lignes.
Le fichier commandes contient 846535 lignes.
Le fichier communes contient 142 lignes.


Le fichier gps contient 5174905 lignes.


1.3) L'identifiant du véhicule n'est pas une colonne des fichiers GPS : il est dans le chemin du fichier. Ajoutez une colonne `vehicule_id` (fonction `F.input_file_name()`), et une colonne `jour` (la date du point).

In [16]:
dataframes["gps"] = dataframes["gps"].withColumn(
    "vehicule_id",
    F.regexp_extract(F.input_file_name(), r"/vehicules/([^/]+)/", 1)
).withColumn(
    "jour",
    F.to_date(F.col("timestamp"))
)

dataframes["gps"].select("vehicule_id", "timestamp", "jour").show(5)

+-----------+-------------------+----------+
|vehicule_id|          timestamp|      jour|
+-----------+-------------------+----------+
|        V18|2025-10-01 07:40:00|2025-10-01|
|        V18|2025-10-01 07:41:00|2025-10-01|
|        V18|2025-10-01 07:42:00|2025-10-01|
|        V18|2025-10-01 07:43:00|2025-10-01|
|        V18|2025-10-01 07:44:00|2025-10-01|
+-----------+-------------------+----------+
only showing top 5 rows



## 2) Vérifier la donnée

2.1) Combien y a-t-il de commandes, de clients, de livreurs et de véhicules ?

In [17]:
#compter le nombre de livreurs distincts dans le dataframe gps
nb_livreurs = dataframes["gps"].select("livreur_id").distinct().count()
nb_clients = dataframes["commandes"].select("client_id").distinct().count()
nb_vehicules = dataframes["gps"].select("vehicule_id").distinct().count()
nb_commandes = dataframes["commandes"].select("commande_id").distinct().count()

resultats = {
    "nb_livreurs": nb_livreurs,
    "nb_clients": nb_clients,
    "nb_vehicules": nb_vehicules,
    "nb_commandes": nb_commandes
}

print("resultats :", resultats)


resultats : {'nb_livreurs': 90, 'nb_clients': 8000, 'nb_vehicules': 72, 'nb_commandes': 271617}


2.2) Comptez les événements du journal par type.

In [18]:
dataframes["commandes"].groupBy("evenement").count().show()

+---------+------+
|evenement| count|
+---------+------+
| commande|271617|
|    echec| 11220|
| reportee|  6960|
|   livree|268682|
|  chargee|286862|
|   retour|  1194|
+---------+------+



2.3) Pour chaque commande, comptez ses événements `chargee`. Combien de commandes ont été chargées 0, 1, 2, 3… fois ?

In [19]:
chargements_par_commande = (
    dataframes["commandes"]
    .groupBy("commande_id")
    .agg(
        F.sum(
            F.when(F.col("evenement") == "chargee", 1).otherwise(0)
        ).alias("nb_chargements")
    )
)

chargements_par_commande.orderBy(
    "nb_chargements", ascending=False
).show(10)


chargements_par_commande.groupBy("nb_chargements").count() \
    .orderBy("nb_chargements").show()

+-----------+--------------+
|commande_id|nb_chargements|
+-----------+--------------+
|   C0022585|             5|
|   C0083318|             4|
|   C0105078|             4|
|   C0026444|             4|
|   C0034971|             4|
|   C0014222|             4|
|   C0118031|             4|
|   C0027754|             4|
|   C0011560|             4|
|   C0055460|             4|
+-----------+--------------+
only showing top 10 rows



+--------------+------+
|nb_chargements| count|
+--------------+------+
|             0|  1637|
|             1|254229|
|             2| 14673|
|             3|  1026|
|             4|    51|
|             5|     1|
+--------------+------+



2.4) Pour chaque événement `chargee`, trouvez l'événement qui le précède dans la vie de la même commande. Comptez les `chargee` par événement précédent.

In [20]:
fenetre = Window.partitionBy("commande_id").orderBy("horodatage")

journal_avec_precedent = dataframes["commandes"].withColumn(
    "evenement_precedent",
    F.lag("evenement").over(fenetre)
)

chargements = journal_avec_precedent.filter(
    F.col("evenement") == "chargee"
)

#chargements.select("evenement", "evenement_precedent").show(10)

chargements.groupBy("evenement_precedent").count().show()

+-------------------+------+
|evenement_precedent| count|
+-------------------+------+
|           commande|269980|
|              echec|  9950|
|           reportee|  6932|
+-------------------+------+



*Un rechargement suit toujours un `echec` (nouvelle tentative) ou un `reportee` (commande rapportée en fin de service) : il y a plus de `chargee` que de `commande`.*

2.5) Au 31/12/2025, combien de commandes ne sont pas livrées ? Séparez les commandes abandonnées (événement `retour`) des commandes encore en cours.

In [21]:
journal_fin_2025 = dataframes["commandes"].filter(
    F.col("horodatage") < F.lit("2026-01-01").cast("timestamp")
)

journal_fin_2025.groupBy("evenement").count().show()

etat_commandes = (
    journal_fin_2025
    .groupBy("commande_id")
    .agg(
        F.sum(
            F.when(F.col("evenement") == "livree", 1).otherwise(0)
        ).alias("nb_livrees"),
        # À toi : même principe pour compter les événements "retour"
        F.sum(
            F.when(F.col("evenement") == "retour", 1).otherwise(0)
        ).alias("nb_retours")
    )
)

non_livrees = etat_commandes.filter(F.col("nb_livrees") == 0)
print("Nombre de commandes non livrées :", non_livrees.count())
#separer les commandes non livrées en deux groupes : celles qui ont été retournées et celles qui n'ont pas été retournées
non_livrees.groupBy("nb_retours").count().orderBy("nb_retours").show()



+---------+------+
|evenement| count|
+---------+------+
| commande|271617|
|    echec| 11220|
| reportee|  6960|
|   livree|268682|
|  chargee|286862|
|   retour|  1194|
+---------+------+

Nombre de commandes non livrées : 2935
+----------+-----+
|nb_retours|count|
+----------+-----+
|         0| 1741|
|         1| 1194|
+----------+-----+



## 3) Les lignes de commande

3.1) Les produits d'une commande sont dans la colonne `detail` de l'événement `commande`. Construisez la table `lignes` : une ligne par produit commandé, avec `commande_id`, `client_id`, `horodatage`, `produit_id`, `quantite`.

In [26]:
lignes = (
    dataframes["commandes"]
    .filter(F.col("evenement") == "commande")

    # on sépare les produits et créer une ligne par produit
    .withColumn("produit", F.explode(F.split(F.col("detail"), r"\|")))

    #on sépare l'identifiant du produit et sa quantité
    .withColumn("produit_id", F.split(F.col("produit"), "x").getItem(0))
    .withColumn(
        "quantite",
        F.split(F.col("produit"), "x").getItem(1).cast("double")
    )

    .select(
        "commande_id", "client_id", "horodatage", "produit_id", "quantite"
    )
)

lignes.show(5)

+-----------+---------+-------------------+----------+--------+
|commande_id|client_id|         horodatage|produit_id|quantite|
+-----------+---------+-------------------+----------+--------+
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0037|     5.0|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0063|     1.0|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0019|     2.0|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0067|     1.0|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0069|     2.0|
+-----------+---------+-------------------+----------+--------+
only showing top 5 rows



3.2) Ajoutez à `lignes` les colonnes du catalogue et le `montant` de chaque ligne. Construisez la table `commandes` : une ligne par commande, avec `montant`, `poids_kg` et `nb_lignes`.

In [27]:
lignes = (
    lignes
    .join(dataframes["produits"], on="produit_id", how="left")
    .withColumn("montant", F.col("prix") * F.col("quantite"))
)

lignes.show(5)

commandes = (
    lignes
    .groupBy("commande_id", "client_id", "horodatage")
    .agg(
        F.sum("montant").alias("montant"),
        F.sum(
            F.col("poids_kg") * F.col("quantite")
        ).alias("poids_kg"),
        F.count("*").alias("nb_lignes")
    )
)

commandes.show(5)

+----------+-----------+---------+-------------------+--------+--------------------+---------+-------------------+----+--------+-------+
|produit_id|commande_id|client_id|         horodatage|quantite|                 nom|categorie|     sous_categorie|prix|poids_kg|montant|
+----------+-----------+---------+-------------------+--------+--------------------+---------+-------------------+----+--------+-------+
|     P0037|   C0000001|  CL05658|2025-01-01 00:03:44|     5.0|Cookies chocolat ...| Épicerie| Biscuits et snacks| 2.6|     0.3|   13.0|
|     P0063|   C0000001|  CL05658|2025-01-01 00:03:44|     1.0| Lardons fumés 200 g|    Frais|Viandes et poissons| 2.4|     0.2|    2.4|
|     P0019|   C0000001|  CL05658|2025-01-01 00:03:44|     2.0|   Capsules café x30| Épicerie|     Petit-déjeuner| 8.5|     0.2|   17.0|
|     P0067|   C0000001|  CL05658|2025-01-01 00:03:44|     1.0|        Bananes 1 kg|    Frais|  Fruits et légumes| 1.9|     1.0|    1.9|
|     P0069|   C0000001|  CL05658|2025-01

+-----------+---------+-------------------+------------------+-----------------+---------+
|commande_id|client_id|         horodatage|           montant|         poids_kg|nb_lignes|
+-----------+---------+-------------------+------------------+-----------------+---------+
|   C0000168|  CL06333|2025-01-01 13:16:07|            209.35|60.97999999999999|       32|
|   C0000469|  CL03079|2025-01-01 22:39:48| 80.89999999999998|            14.03|       13|
|   C0000752|  CL00230|2025-01-02 16:34:53|            218.05|           46.515|       31|
|   C0000810|  CL07244|2025-01-02 18:29:22|239.45000000000002|78.49999999999999|       27|
|   C0000855|  CL04907|2025-01-02 19:48:36| 70.89999999999999|            15.87|       15|
+-----------+---------+-------------------+------------------+-----------------+---------+
only showing top 5 rows



## 4) Les entrepôts

4.1) Pour chaque véhicule et chaque jour, gardez le premier et le dernier point GPS. Arrondissez leurs coordonnées à 0,01° et comptez les journées par position arrondie. Combien d'entrepôts y a-t-il ?

In [28]:
points_jour = (
    dataframes["gps"]
    .groupBy("vehicule_id", "jour")
    .agg(
        F.min(F.struct("timestamp", "lon", "lat")).alias("premier"),
        F.max(F.struct("timestamp", "lon", "lat")).alias("dernier")
    )
    .cache()
)

premiers_points = points_jour.select(
    "vehicule_id",
    "jour",
    F.col("premier.lon").alias("lon"),
    F.col("premier.lat").alias("lat")
)

derniers_points = points_jour.select(
    "vehicule_id",
    "jour",
    F.col("dernier.lon").alias("lon"),
    F.col("dernier.lat").alias("lat")
)

# Réunir les départs et les retours, puis arrondir à 0,01°.
positions_jour = (
    premiers_points.unionByName(derniers_points)
    .withColumn("lon_arrondie", F.round("lon", 2))
    .withColumn("lat_arrondie", F.round("lat", 2))
    .select("vehicule_id", "jour", "lon_arrondie", "lat_arrondie")
    .dropDuplicates()
)

positions_entrepots = (
    positions_jour
    .groupBy("lon_arrondie", "lat_arrondie")
    .count()
    .withColumnRenamed("count", "nb_journees")
)

print(f"Nombre de journées de véhicules analysées : {points_jour.count():,}")

print("\nPositions de départ ou de retour, classées par fréquentation :")
positions_entrepots.orderBy(
    F.desc("nb_journees")
).show(truncate=False)

print(f"Nombre de positions d'entrepôts identifiées : {positions_entrepots.count()}")

Nombre de journées de véhicules analysées : 10,392

Positions de départ ou de retour, classées par fréquentation :


+------------+------------+-----------+
|lon_arrondie|lat_arrondie|nb_journees|
+------------+------------+-----------+
|2.45        |48.91       |4217       |
|2.28        |48.94       |3208       |
|2.35        |48.75       |2967       |
+------------+------------+-----------+



Nombre de positions d'entrepôts identifiées : 3


4.2) Construisez la table `entrepots` (`entrepot`, `lon`, `lat`) : `E1`, `E2`… par longitude croissante, position = moyenne des premiers points. Construisez la table `vehicule_entrepot` (`vehicule_id`, `entrepot`) et comptez les véhicules par entrepôt.

In [29]:
departs = (
    premiers_points
    .withColumn("lon_arrondie", F.round("lon", 2))
    .withColumn("lat_arrondie", F.round("lat", 2))
)

# Position réelle de chaque entrepôt = moyenne des premiers points.
entrepots_positions = (
    departs
    .groupBy("lon_arrondie", "lat_arrondie")
    .agg(
        F.avg("lon").alias("lon"),
        F.avg("lat").alias("lat")
    )
)

# Attribuer E1, E2... par longitude croissante.
fenetre_entrepots = Window.orderBy("lon", "lat")

entrepots_positions = (
    entrepots_positions
    .withColumn(
        "entrepot",
        F.concat(
            F.lit("E"),
            F.row_number().over(fenetre_entrepots).cast("string")
        )
    )
    .cache()
)

entrepots = entrepots_positions.select("entrepot", "lon", "lat")

# Relier chaque véhicule à son entrepôt grâce à la position de départ.
vehicule_entrepot = (
    departs
    .select("vehicule_id", "lon_arrondie", "lat_arrondie")
    .distinct()
    .join(
        entrepots_positions.select(
            "lon_arrondie", "lat_arrondie", "entrepot"
        ),
        on=["lon_arrondie", "lat_arrondie"],
        how="left"
    )
    .select("vehicule_id", "entrepot")
    .distinct()
)

print("Entrepôts et coordonnées moyennes, par longitude croissante :")
entrepots.orderBy("lon").show(truncate=False)

print("\nNombre de véhicules par entrepôt :")
(
    vehicule_entrepot
    .groupBy("entrepot")
    .agg(F.countDistinct("vehicule_id").alias("nb_vehicules"))
    .orderBy("entrepot")
    .show(truncate=False)
)

print("\nExemple d'affectation des véhicules aux entrepôts :")
vehicule_entrepot.orderBy("vehicule_id").show(10, truncate=False)

15:59:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:59:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:59:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:59:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:59:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Entrepôts et coordonnées moyennes, par longitude croissante :


15:59:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:59:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:59:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
15:59:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+------------------+------------------+
|entrepot|lon               |lat               |
+--------+------------------+------------------+
|E1      |2.2770689133416475|48.93671290461345 |
|E2      |2.3515135942028986|48.74704698584426 |
|E3      |2.4489591059995255|48.906701645482585|
+--------+------------------+------------------+


Nombre de véhicules par entrepôt :


+--------+------------+
|entrepot|nb_vehicules|
+--------+------------+
|E1      |24          |
|E2      |24          |
|E3      |24          |
+--------+------------+


Exemple d'affectation des véhicules aux entrepôts :


+-----------+--------+
|vehicule_id|entrepot|
+-----------+--------+
|V01        |E1      |
|V02        |E1      |
|V03        |E1      |
|V04        |E1      |
|V05        |E1      |
|V06        |E1      |
|V07        |E1      |
|V08        |E1      |
|V09        |E1      |
|V10        |E1      |
+-----------+--------+
only showing top 10 rows



## 5) Les livraisons

On découpe la journée de chaque livreur en **livraisons** : une livraison commence quand le livreur descend du véhicule et dure jusqu'à la livraison suivante. Le chargement du matin, les retours à l'entrepôt et la pause déjeuner sont aussi des périodes à pied : la question 5.6 les distingue.

5.1) Ajoutez à `gps` une colonne `changement` : 1 si le livreur vient de descendre du véhicule (point précédent du même livreur, le même jour, `dans_vehicule = true` ; point courant `false`), 0 sinon. Le premier point de la journée vaut 0.

Affichez les lignes où `changement = 1` pour le livreur `L001` le 2025-01-06.

In [31]:
# Ordonner les points de chaque livreur, pour chaque journée.
fenetre_gps = (
    Window
    .partitionBy("livreur_id", "jour")
    .orderBy("timestamp")
)

dataframes["gps"] = (
    dataframes["gps"]
    .withColumn(
        "dans_vehicule_avant",
        F.lag("dans_vehicule").over(fenetre_gps)
    )
    .withColumn(
        "changement",
        F.when(
            (F.col("dans_vehicule_avant") == True)
            & (F.col("dans_vehicule") == False),
            1
        ).otherwise(0)
    )
    .drop("dans_vehicule_avant")
)

# Afficher les descentes du véhicule pour le livreur et le jour demandés.
descentes = dataframes["gps"].filter(
    (F.col("livreur_id") == "L001")
    & (F.col("jour") == F.lit("2025-01-06").cast("date"))
    & (F.col("changement") == 1)
)

print("Descentes du véhicule de L001 le 6 janvier 2025 :")

descentes.select(
    "livreur_id",
    "vehicule_id",
    "timestamp",
    "dans_vehicule",
    "changement"
).orderBy("timestamp").show(100, truncate=False)

Descentes du véhicule de L001 le 6 janvier 2025 :


+----------+-----------+-------------------+-------------+----------+
|livreur_id|vehicule_id|timestamp          |dans_vehicule|changement|
+----------+-----------+-------------------+-------------+----------+
|L001      |V10        |2025-01-06 09:54:00|false        |1         |
|L001      |V10        |2025-01-06 10:00:00|false        |1         |
|L001      |V10        |2025-01-06 10:09:00|false        |1         |
|L001      |V10        |2025-01-06 10:23:00|false        |1         |
|L001      |V10        |2025-01-06 10:36:00|false        |1         |
|L001      |V10        |2025-01-06 10:54:00|false        |1         |
|L001      |V10        |2025-01-06 11:12:00|false        |1         |
|L001      |V10        |2025-01-06 11:38:00|false        |1         |
|L001      |V10        |2025-01-06 12:02:00|false        |1         |
|L001      |V10        |2025-01-06 12:15:00|false        |1         |
|L001      |V10        |2025-01-06 13:04:00|false        |1         |
|L001      |V10     

5.2) Ajoutez une colonne `numero_livraison` : la somme cumulée de `changement` pour chaque livreur et chaque jour. Elle vaut 0 au chargement du matin.

Pour `L001` le 2025-01-06, affichez par `numero_livraison` le premier et le dernier timestamp, et le nombre de minutes à pied.

In [32]:
# Fenêtre cumulative : du premier point au point courant.
fenetre_cumulative = (
    Window
    .partitionBy("livreur_id", "jour")
    .orderBy("timestamp")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

dataframes["gps"] = dataframes["gps"].withColumn(
    "numero_livraison",
    F.sum("changement").over(fenetre_cumulative)
)

# Sélectionner la journée demandée.
gps_l001 = dataframes["gps"].filter(
    (F.col("livreur_id") == "L001")
    & (F.col("jour") == F.lit("2025-01-06").cast("date"))
)

# Résumer chaque période.
resume_livraisons = (
    gps_l001
    .groupBy("numero_livraison")
    .agg(
        F.min("timestamp").alias("premier_timestamp"),
        F.max("timestamp").alias("dernier_timestamp"),
        F.sum(
            F.when(F.col("dans_vehicule") == False, 1).otherwise(0)
        ).alias("minutes_a_pied")
    )
    .orderBy("numero_livraison")
)

print("Périodes de L001 le 6 janvier 2025 :")
print("Le numéro 0 correspond au chargement du matin.")
print("Chaque point GPS à pied représente une minute.")

resume_livraisons.show(100, truncate=False)

Périodes de L001 le 6 janvier 2025 :
Le numéro 0 correspond au chargement du matin.
Chaque point GPS à pied représente une minute.


+----------------+-------------------+-------------------+--------------+
|numero_livraison|premier_timestamp  |dernier_timestamp  |minutes_a_pied|
+----------------+-------------------+-------------------+--------------+
|0               |2025-01-06 07:43:00|2025-01-06 09:53:00|44            |
|1               |2025-01-06 09:54:00|2025-01-06 09:59:00|4             |
|2               |2025-01-06 10:00:00|2025-01-06 10:08:00|7             |
|3               |2025-01-06 10:09:00|2025-01-06 10:22:00|10            |
|4               |2025-01-06 10:23:00|2025-01-06 10:35:00|4             |
|5               |2025-01-06 10:36:00|2025-01-06 10:53:00|8             |
|6               |2025-01-06 10:54:00|2025-01-06 11:11:00|4             |
|7               |2025-01-06 11:12:00|2025-01-06 11:37:00|18            |
|8               |2025-01-06 11:38:00|2025-01-06 12:01:00|17            |
|9               |2025-01-06 12:02:00|2025-01-06 12:14:00|4             |
|10              |2025-01-06 12:15:00|

5.3) Construisez la table `livraisons` : une ligne par (`livreur_id`, `jour`, `numero_livraison`), calculée sur les seuls points où le livreur est à pied, avec `vehicule_id`, `debut`, `fin`, `duree_min` (nombre de points) et la position moyenne `lon`, `lat`. Combien de lignes contient-elle ?

In [33]:
livraisons = (
    dataframes["gps"]
    .filter(F.col("dans_vehicule") == False)
    .groupBy("livreur_id", "jour", "numero_livraison")
    .agg(
        F.first("vehicule_id", ignorenulls=True).alias("vehicule_id"),
        F.min("timestamp").alias("debut"),
        F.max("timestamp").alias("fin"),
        F.count("*").alias("duree_min"),
        F.avg("lon").alias("lon"),
        F.avg("lat").alias("lat")
    )
)

print(f"Nombre de périodes à pied : {livraisons.count():,}")
print("La durée correspond au nombre de points GPS à pied.")

print("\nExemple pour L001 le 6 janvier 2025 :")
(
    livraisons
    .filter(
        (F.col("livreur_id") == "L001")
        & (F.col("jour") == F.lit("2025-01-06").cast("date"))
    )
    .orderBy("numero_livraison")
    .show(100, truncate=False)
)

Nombre de périodes à pied : 273,593
La durée correspond au nombre de points GPS à pied.

Exemple pour L001 le 6 janvier 2025 :


+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+
|livreur_id|jour      |numero_livraison|vehicule_id|debut              |fin                |duree_min|lon               |lat               |
+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+
|L001      |2025-01-06|0               |V10        |2025-01-06 07:43:00|2025-01-06 08:26:00|44       |2.277131022727271 |48.93670315909091 |
|L001      |2025-01-06|1               |V10        |2025-01-06 09:54:00|2025-01-06 09:57:00|4        |2.311953          |48.87138975       |
|L001      |2025-01-06|2               |V10        |2025-01-06 10:00:00|2025-01-06 10:06:00|7        |2.313446714285714 |48.87084757142858 |
|L001      |2025-01-06|3               |V10        |2025-01-06 10:09:00|2025-01-06 10:18:00|10       |2.3160128999999996|48.87292010000001 |
|L001      |2

5.4) Ajoutez à chaque livraison ses signatures et ses échecs. Un événement `livree` ou `echec` appartient à une livraison quand il a le même livreur, le même jour, et un horodatage compris entre `debut` et `fin`, avec 1 minute de marge de chaque côté : les points GPS sont à la minute, les événements à la seconde.

Construisez la table `livraisons_ev` : les colonnes de `livraisons`, plus `signature` (horodatage de l'événement), `evenement`, `commande_id` et `client_id`. Une ligne par événement ; une livraison sans événement garde une ligne, avec ces 4 colonnes vides.

In [34]:
# Garder uniquement les signatures et les échecs du journal.
evenements_livraison = (
    dataframes["commandes"]
    .filter(F.col("evenement").isin("livree", "echec"))
    .select(
        "livreur_id",
        "commande_id",
        "client_id",
        "evenement",
        "horodatage",
        F.to_date("horodatage").alias("jour")
    )
)

l = livraisons.alias("l")
e = evenements_livraison.alias("e")

# Même livreur, même jour, événement dans la période avec ±1 minute.
condition_rattachement = (
    (F.col("l.livreur_id") == F.col("e.livreur_id"))
    & (F.col("l.jour") == F.col("e.jour"))
    & F.col("e.horodatage").between(
        F.col("l.debut") - F.expr("INTERVAL 1 MINUTE"),
        F.col("l.fin") + F.expr("INTERVAL 1 MINUTE")
    )
)

livraisons_ev = (
    l.join(e, condition_rattachement, how="left")
    .select(
        "l.*",
        F.col("e.horodatage").alias("signature"),
        F.col("e.evenement").alias("evenement"),
        F.col("e.commande_id").alias("commande_id"),
        F.col("e.client_id").alias("client_id")
    )
)

print("Exemple des événements associés à L001 le 6 janvier 2025 :")

(
    livraisons_ev
    .filter(
        (F.col("livreur_id") == "L001")
        & (F.col("jour") == F.lit("2025-01-06").cast("date"))
    )
    .select(
        "numero_livraison", "debut", "fin",
        "signature", "evenement", "commande_id", "client_id"
    )
    .orderBy("numero_livraison", "signature")
    .show(30, truncate=False)
)

Exemple des événements associés à L001 le 6 janvier 2025 :


+----------------+-------------------+-------------------+-------------------+---------+-----------+---------+
|numero_livraison|debut              |fin                |signature          |evenement|commande_id|client_id|
+----------------+-------------------+-------------------+-------------------+---------+-----------+---------+
|0               |2025-01-06 07:43:00|2025-01-06 08:26:00|NULL               |NULL     |NULL       |NULL     |
|1               |2025-01-06 09:54:00|2025-01-06 09:57:00|2025-01-06 09:54:50|livree   |C0001606   |CL07994  |
|2               |2025-01-06 10:00:00|2025-01-06 10:06:00|2025-01-06 10:02:25|livree   |C0001917   |CL02560  |
|3               |2025-01-06 10:09:00|2025-01-06 10:18:00|2025-01-06 10:13:28|livree   |C0002443   |CL05981  |
|4               |2025-01-06 10:23:00|2025-01-06 10:26:00|2025-01-06 10:24:48|livree   |C0001770   |CL04561  |
|5               |2025-01-06 10:36:00|2025-01-06 10:43:00|2025-01-06 10:42:05|livree   |C0002251   |CL05834  |
|

5.5) Combien d'événements `livree` et `echec` du journal ne sont rattachés à aucune livraison ?

In [35]:
evenements_non_rattaches = e.join(
    l,
    condition_rattachement,
    how="left_anti"
)

print("Événements du journal non rattachés à une livraison :")

evenements_non_rattaches.agg(
    F.count(
        F.when(F.col("evenement") == "livree", 1)
    ).alias("livree_non_rattaches"),

    F.count(
        F.when(F.col("evenement") == "echec", 1)
    ).alias("echec_non_rattaches"),

    F.count("*").alias("total_non_rattaches")
).show(truncate=False)

Événements du journal non rattachés à une livraison :


+--------------------+-------------------+-------------------+
|livree_non_rattaches|echec_non_rattaches|total_non_rattaches|
+--------------------+-------------------+-------------------+
|0                   |0                  |0                  |
+--------------------+-------------------+-------------------+



5.6) Toutes les périodes à pied ne sont pas des livraisons. Ajoutez à `livraisons_ev` une colonne `type` :
- `livraison` : au moins un événement ;
- `entrepot` : sinon, position à moins de 300 m d'un entrepôt ;
- `pause` : les autres.

Pour chaque type : nombre de lignes de `livraisons`, durée moyenne, heure moyenne de début.

5.7) Une même livraison peut compter plusieurs événements : plusieurs commandes à la même adresse, remises à pied l'une après l'autre. Combien de livraisons ont 1, 2, 3… événements ? Combien concernent au moins 2 clients différents ?

5.8) Ajoutez une colonne `duree_ev_min` à chaque événement : le temps écoulé depuis l'événement précédent de la même livraison, ou depuis le début de la livraison pour le premier. Quelle est la durée moyenne d'une remise signée ? D'un échec ?

5.9) Pour chaque livreur : nombre de livraisons signées, nombre d'échecs, nombre de jours travaillés, livraisons signées par jour. Affichez les 10 livreurs qui signent le plus de livraisons par jour.

## 6) Les distances

6.1) Ajoutez à `gps` une colonne `distance_km` : la distance depuis le point précédent du même livreur, le même jour (`haversine`). Le premier point de la journée vaut 0.

*Un point par minute coupe les virages : la distance calculée sous-estime d'environ 12 % la distance réellement parcourue.*

6.2) Kilomètres parcourus sur l'année par chaque véhicule : les 10 premiers, et le total de la flotte.

6.3) Kilomètres parcourus sur l'année par chaque livreur : les 10 premiers.

6.4) Kilomètres parcourus par jour : moyenne par véhicule-jour, pour chaque entrepôt.

6.5) Distance entre deux livraisons : la somme de `distance_km` de chaque (`livreur_id`, `jour`, `numero_livraison`), c'est-à-dire le trajet qui suit la livraison jusqu'à la suivante. Moyenne et médiane, pour les livraisons de type `livraison`.

## 7) Les ventes

7.1) Chiffre d'affaires par mois (mois de la commande).

7.2) Panier moyen : montant moyen d'une commande.

7.3) Panier médian.

7.4) Top 10 des produits par chiffre d'affaires.

7.5) Top 10 des produits par quantité vendue.

7.6) Top 10 des clients par dépense sur l'année.

7.7) Top 10 des clients par nombre de commandes.

## 8) La qualité de service

8.1) Délai entre la commande et sa livraison, en jours calendaires (date de la signature − date de la commande). Combien de commandes livrées pour chaque délai ?

8.2) Part des commandes livrées le lendemain, pour chaque mois de commande.

8.3) Construisez la table `suivi` : une ligne par commande, avec `nb_tentatives` (événements `livree` + `echec`), `nb_echecs` et `issue` (`livree`, `retour` ou `en cours`). Combien de commandes pour chaque nombre de tentatives ?

8.4) Motifs d'échec : nombre d'échecs par motif.

8.5) Top 10 des clients au plus fort taux d'échec (échecs / tentatives), parmi les clients qui ont au moins 10 tentatives.

## 9) Les clients

On construit la table `clients` colonne par colonne : une ligne par client qui a passé au moins une commande.

9.1) Construisez la table `clients` : `client_id`, `nb_commandes`, `ca` (somme des montants), `panier_moyen`.

9.2) Ajoutez `premiere_commande` et `derniere_commande` (dates).

9.3) Ajoutez `lon` et `lat` : la position moyenne des livraisons où le client a signé. Combien de clients n'ont aucune position ?

9.4) Ajoutez `commune` et `code_postal` : la commune dont le centre (`communes.csv`) est le plus proche de la position du client.

9.5) Les 10 communes qui passent le plus de commandes, avec leur chiffre d'affaires.

9.6) Ajoutez `tentatives`, `echecs` et `taux_echec` (à partir de `suivi`, 8.3).

9.7) Ajoutez `categorie_dominante` (la catégorie où le client dépense le plus) et `part_dominante` (part de sa dépense dans cette catégorie).

9.8) Ajoutez `unites_par_ligne` : la quantité moyenne par ligne de commande.

9.9) Segments de dépense annuelle (`ca`) : moins de 1 000 €, 1 000 à 5 000 €, 5 000 à 20 000 €, 20 000 à 50 000 €, 50 000 € et plus. Ajoutez la colonne `segment`, puis pour chaque segment : nombre de clients, chiffre d'affaires, chiffre d'affaires moyen par client, part du chiffre d'affaires total.

9.10) Nouveaux clients par mois (mois de la première commande), en tableau et en graphique.

9.11) Pour chaque catégorie dominante : nombre de clients et chiffre d'affaires.

9.12) Clients atypiques : les 10 clients dont `unites_par_ligne` est la plus élevée, avec le rapport à la moyenne de tous les clients, leur `ca` et leur `categorie_dominante`.

## 10) Data visualisation

Échelles de couleur : blanc ou clair pour 0, foncé pour les valeurs élevées.

10.1) Heatmap jour de la semaine × heure du nombre de commandes passées.

10.2) Carte du nombre de livraisons signées par commune ou arrondissement (`communes.geojson`, carte choroplèthe).

10.3) Heatmap heure × mois de la vitesse moyenne quand le livreur est à bord et que le véhicule roule.

10.4) Treemap du chiffre d'affaires : catégorie → sous-catégorie.

## 11) Sauvegarde et modélisation

11.2) **Modélisation.** Les analyses précédentes recalculent les mêmes jointures et agrégations. Proposez des collections MongoDB qui rendent ces questions immédiates : pour chaque collection, ce que représente un document, ses champs, ses tableaux imbriqués, et les questions des parties 5 à 9 qu'elle simplifie. Écrivez votre proposition dans la cellule ci-dessous.

*Votre proposition :*